# Step 1 — Audit & Design of the Synthetic Multimodal Dataset

This notebook uses the existing `multimodal_patient_year(1).csv` as the reference dataset.

It creates:
- a full column audit,
- an encoder map,
- leakage warnings,
- a proposed Version-1 synthetic dataset schema,
- dataset summary files.

**Model plan (V1):**

`Static valve features → Valve Encoder (MLP)`  
`Labs/hemodynamics over time → Physiology Encoder (GRU)`  
`Medications over time → Medication Encoder (GRU)`  
`Temporal fusion + valve embedding → Survival Head → S(t)`

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

# Change this only if your CSV is stored elsewhere.
INPUT_PATH = Path("multimodal_patient_year.csv")
OUTDIR = Path('/mnt/data/synthetic_design_outputs')
OUTDIR.mkdir(parents=True, exist_ok=True)

print('Input:', INPUT_PATH)
print('Output directory:', OUTDIR)

Input: multimodal_patient_year.csv
Output directory: \mnt\data\synthetic_design_outputs


## 1. Rules, feature groups, and helper functions

In [2]:
# ---------------------------------------------------------------------
# 1. Column classification rules
# ---------------------------------------------------------------------

TARGET_COLUMNS = {
    "Valve_Failure",
}

IDENTIFIER_TIME_COLUMNS = {
    "Patient",
    "Year",
    "Index_Date",
    "relative_to_index_year_UNVERIFIED",
}

NOTE_METADATA_COLUMNS = {
    "note_count",
    "note_types",
    "note_services",
    "signed_statuses",
    "provider_types",
    "provider_specialties",
    "min_creation_year",
    "max_creation_year",
    "max_last_edited_year",
    "notes_observed",
    "notes_available",
}

RAW_TEXT_COLUMNS = {
    "note_text",
}

# High-risk leakage columns: should NOT be blindly fed to the model.
# They can contain information that is effectively the outcome itself
# or happens after the deterioration/failure event.
POTENTIAL_LEAKAGE_COLUMNS = {
    "Valve_Failure",
    "note_mentions_prosthetic_failure",
    "note_mentions_redo",
    "note_mentions_valve_in_valve",
}

# Note-derived features potentially useful in V1 if they are temporally
# restricted to information available BEFORE the prediction time.
NOTE_DERIVED_PREFIX = "note_mentions_"
LAB_PREFIX = "lab__"
MED_PREFIX = "med__"

MED_METADATA_COLUMNS = {
    "med_rows",
    "med_weighted_records",
    "med_unique_generic",
    "med_unique_therapeutic_classes",
    "med_inpatient_rows",
    "med_outpatient_rows",
    "medications_observed",
    "medications_available",
}

LAB_METADATA_COLUMNS = {
    "labs_observed",
    "labs_available",
}

OTHER_QC_COLUMNS = {
    "rich_lab_med",
}


def classify_column(col: str):
    """Return modality, role, encoder, V1 recommendation, and leakage note."""

    if col in TARGET_COLUMNS:
        return (
            "outcome",
            "target",
            "NONE",
            "TARGET_ONLY",
            "Never use as encoder input.",
        )

    if col in IDENTIFIER_TIME_COLUMNS:
        if col == "Patient":
            role = "identifier"
        else:
            role = "time/index"
        return (
            "index",
            role,
            "NONE",
            "KEEP",
            "Used for grouping, ordering, censor/event timing; not an embedding feature by default.",
        )

    if col in RAW_TEXT_COLUMNS:
        return (
            "notes",
            "raw_text",
            "Text Encoder (V2)",
            "KEEP_FOR_V2",
            "Do not use in V1. Retain for future Clinical Text Encoder.",
        )

    if col in NOTE_METADATA_COLUMNS:
        return (
            "notes",
            "metadata",
            "Optional / none",
            "REVIEW",
            "Mostly QC/availability metadata; not core clinical signal.",
        )

    if col.startswith(NOTE_DERIVED_PREFIX):
        if col in POTENTIAL_LEAKAGE_COLUMNS:
            return (
                "notes",
                "note_derived",
                "Valve/Temporal Encoder only if baseline-safe",
                "EXCLUDE_OR_TEMPORALLY_FILTER",
                "Potential label/post-event leakage. Use only if proven available before prediction time.",
            )
        return (
            "notes",
            "note_derived",
            "Valve Encoder or Temporal branch",
            "KEEP_WITH_TEMPORAL_RULE",
            "Use only information available at or before the prediction time.",
        )

    if col.startswith(LAB_PREFIX):
        return (
            "physiology",
            "dynamic",
            "Physiology Encoder (GRU)",
            "KEEP",
            "Longitudinal feature.",
        )

    if col in LAB_METADATA_COLUMNS:
        return (
            "physiology",
            "availability/QC",
            "Mask / auxiliary",
            "KEEP_AS_MASK",
            "Useful for missingness/observation mask rather than core physiology embedding.",
        )

    if col.startswith(MED_PREFIX):
        return (
            "medications",
            "dynamic",
            "Medication Encoder (GRU)",
            "KEEP",
            "Longitudinal medication class indicator.",
        )

    if col in MED_METADATA_COLUMNS:
        return (
            "medications",
            "metadata/QC",
            "Mask / auxiliary",
            "REVIEW",
            "Can support observation masks or utilization context.",
        )

    if col in OTHER_QC_COLUMNS:
        return (
            "qc",
            "quality_flag",
            "NONE",
            "REVIEW",
            "Dataset quality/availability flag.",
        )

    return (
        "unclassified",
        "review",
        "TBD",
        "REVIEW",
        "Manual review required.",
    )


# ---------------------------------------------------------------------
# 2. Proposed NEW variables for the synthetic V1 dataset
# ---------------------------------------------------------------------

ADDITIONAL_SYNTHETIC_VARIABLES = [
    # Core time / survival design
    {
        "variable": "time_since_implant_months",
        "modality": "index",
        "role": "time",
        "encoder": "NONE",
        "source": "ADD_SYNTHETIC",
        "reason": "Canonical longitudinal clock for all encoders.",
    },
    {
        "variable": "implant_date",
        "modality": "valve_static",
        "role": "static",
        "encoder": "Valve Encoder (MLP)",
        "source": "ADD_SYNTHETIC",
        "reason": "Needed to anchor valve age/durability.",
    },
    {
        "variable": "procedure_type",
        "modality": "valve_static",
        "role": "static_categorical",
        "encoder": "Valve Encoder (MLP/embedding)",
        "source": "ADD_SYNTHETIC",
        "reason": "e.g. TAVR vs SAVR.",
    },
    {
        "variable": "valve_model",
        "modality": "valve_static",
        "role": "static_categorical",
        "encoder": "Valve Encoder (MLP/embedding)",
        "source": "ADD_SYNTHETIC",
        "reason": "Valve-specific durability context.",
    },
    {
        "variable": "valve_size_mm",
        "modality": "valve_static",
        "role": "static_numeric",
        "encoder": "Valve Encoder (MLP)",
        "source": "ADD_SYNTHETIC",
        "reason": "Important prosthesis characteristic.",
    },
    {
        "variable": "valve_position",
        "modality": "valve_static",
        "role": "static_categorical",
        "encoder": "Valve Encoder (MLP/embedding)",
        "source": "ADD_SYNTHETIC",
        "reason": "Anatomical context.",
    },

    # Valve hemodynamics
    {
        "variable": "mean_gradient_mmHg",
        "modality": "physiology",
        "role": "dynamic",
        "encoder": "Physiology Encoder (GRU)",
        "source": "ADD_SYNTHETIC",
        "reason": "Core valve deterioration trajectory.",
    },
    {
        "variable": "peak_velocity_m_s",
        "modality": "physiology",
        "role": "dynamic",
        "encoder": "Physiology Encoder (GRU)",
        "source": "ADD_SYNTHETIC",
        "reason": "Core valve hemodynamic marker.",
    },
    {
        "variable": "effective_orifice_area_cm2",
        "modality": "physiology",
        "role": "dynamic",
        "encoder": "Physiology Encoder (GRU)",
        "source": "ADD_SYNTHETIC",
        "reason": "Should decline in stenotic deterioration patterns.",
    },
    {
        "variable": "regurgitation_grade",
        "modality": "physiology",
        "role": "dynamic_ordinal",
        "encoder": "Physiology Encoder (GRU)",
        "source": "ADD_SYNTHETIC",
        "reason": "Captures regurgitation-dominant deterioration.",
    },

    # Optional structured note-derived symptoms/events
    {
        "variable": "note_dyspnea",
        "modality": "notes",
        "role": "dynamic_note_derived",
        "encoder": "Temporal branch / optional",
        "source": "ADD_SYNTHETIC",
        "reason": "Synthetic structured note signal; must be generated from current clinical state.",
    },
    {
        "variable": "note_valve_dysfunction",
        "modality": "notes",
        "role": "dynamic_note_derived",
        "encoder": "Temporal branch / optional",
        "source": "ADD_SYNTHETIC",
        "reason": "May be useful but must be pre-event only to avoid leakage.",
    },

    # Survival target
    {
        "variable": "duration_months",
        "modality": "outcome",
        "role": "survival_target",
        "encoder": "NONE",
        "source": "ADD_SYNTHETIC",
        "reason": "Time from implant to event or censoring.",
    },
    {
        "variable": "event",
        "modality": "outcome",
        "role": "survival_target",
        "encoder": "NONE",
        "source": "ADD_SYNTHETIC",
        "reason": "1 = deterioration/failure event, 0 = censored.",
    },
    {
        "variable": "event_type",
        "modality": "outcome",
        "role": "optional_target",
        "encoder": "NONE",
        "source": "ADD_SYNTHETIC",
        "reason": "Optional subtype: stenotic, regurgitant, reintervention, etc.",
    },
]


# ---------------------------------------------------------------------
# 3. Audit helpers
# ---------------------------------------------------------------------

def safe_numeric_stats(series: pd.Series):
    if not pd.api.types.is_numeric_dtype(series):
        return {
            "mean": np.nan,
            "std": np.nan,
            "min": np.nan,
            "p25": np.nan,
            "median": np.nan,
            "p75": np.nan,
            "max": np.nan,
        }

    clean = series.dropna()
    if clean.empty:
        return {
            "mean": np.nan,
            "std": np.nan,
            "min": np.nan,
            "p25": np.nan,
            "median": np.nan,
            "p75": np.nan,
            "max": np.nan,
        }

    q = clean.quantile([0.25, 0.50, 0.75])
    return {
        "mean": float(clean.mean()),
        "std": float(clean.std()) if len(clean) > 1 else 0.0,
        "min": float(clean.min()),
        "p25": float(q.loc[0.25]),
        "median": float(q.loc[0.50]),
        "p75": float(q.loc[0.75]),
        "max": float(clean.max()),
    }


def make_audit(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for col in df.columns:
        modality, role, encoder, recommendation, warning = classify_column(col)
        stats = safe_numeric_stats(df[col])

        rows.append({
            "column": col,
            "dtype": str(df[col].dtype),
            "non_missing_n": int(df[col].notna().sum()),
            "missing_n": int(df[col].isna().sum()),
            "missing_pct": round(float(df[col].isna().mean() * 100), 2),
            "n_unique": int(df[col].nunique(dropna=True)),
            "modality": modality,
            "role": role,
            "recommended_encoder": encoder,
            "v1_action": recommendation,
            "leakage_or_design_note": warning,
            **stats,
        })

    return pd.DataFrame(rows)


def make_encoder_map(audit: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "column",
        "modality",
        "role",
        "recommended_encoder",
        "v1_action",
        "leakage_or_design_note",
    ]
    return audit[cols].copy()


def proposed_schema_from_existing(audit: pd.DataFrame) -> pd.DataFrame:
    existing = audit.copy()

    def schema_source(action):
        if action in {"KEEP", "KEEP_AS_MASK", "KEEP_WITH_TEMPORAL_RULE", "KEEP_FOR_V2"}:
            return "FROM_MULTIMODAL"
        if action == "TARGET_ONLY":
            return "FROM_MULTIMODAL_TARGET"
        return "REVIEW_EXISTING"

    existing_schema = pd.DataFrame({
        "variable": existing["column"],
        "modality": existing["modality"],
        "role": existing["role"],
        "encoder": existing["recommended_encoder"],
        "source": existing["v1_action"].map(schema_source),
        "reason": existing["leakage_or_design_note"],
    })

    added = pd.DataFrame(ADDITIONAL_SYNTHETIC_VARIABLES)

    combined = pd.concat([existing_schema, added], ignore_index=True)

    # Remove exact duplicate variable definitions, prioritizing newly-added
    # explicit definitions over generic existing ones if names ever overlap.
    combined["_priority"] = np.where(combined["source"] == "ADD_SYNTHETIC", 0, 1)
    combined = (
        combined.sort_values(["variable", "_priority"])
        .drop_duplicates(subset=["variable"], keep="first")
        .drop(columns="_priority")
        .reset_index(drop=True)
    )

    return combined


def dataset_summary(df: pd.DataFrame, audit: pd.DataFrame):
    patient_col = "Patient" if "Patient" in df.columns else None
    year_col = "Year" if "Year" in df.columns else None

    summary = {
        "n_rows": int(len(df)),
        "n_columns": int(df.shape[1]),
        "n_patients": int(df[patient_col].nunique()) if patient_col else None,
        "year_min": int(df[year_col].min()) if year_col else None,
        "year_max": int(df[year_col].max()) if year_col else None,
        "modalities": audit["modality"].value_counts().to_dict(),
        "v1_actions": audit["v1_action"].value_counts().to_dict(),
        "potential_leakage_columns_present": [
            c for c in POTENTIAL_LEAKAGE_COLUMNS if c in df.columns
        ],
    }

    if "Valve_Failure" in df.columns:
        target = df["Valve_Failure"].dropna()
        summary["Valve_Failure_counts"] = {
            str(k): int(v) for k, v in target.value_counts().to_dict().items()
        }

    return summary


def write_next_steps(path: Path):
    text = """NEXT STEPS — VERSION 1

1. REVIEW 01_column_audit.csv
   - Confirm which existing variables are clinically meaningful.
   - Pay special attention to columns marked EXCLUDE_OR_TEMPORALLY_FILTER.

2. REVIEW 02_encoder_map.csv
   - Static valve features -> Valve Encoder (MLP).
   - lab__* -> Physiology Encoder (GRU).
   - med__* -> Medication Encoder (GRU).
   - event/duration -> targets only, never encoder inputs.

3. REVIEW 03_proposed_synthetic_schema.csv
   Add the valve-specific variables that are missing from the current multimodal:
   - implant_date
   - procedure_type
   - valve_model
   - valve_size_mm
   - valve_position
   - time_since_implant_months
   - mean_gradient_mmHg
   - peak_velocity_m_s
   - effective_orifice_area_cm2
   - regurgitation_grade
   - duration_months
   - event

4. DO NOT TRAIN ANY ENCODER YET.
   First we must define:
   - clinically plausible ranges,
   - baseline correlations,
   - longitudinal trajectories,
   - event-generation rules,
   - censoring rules,
   - realistic missingness.

5. CLINICAL NOTES — VERSION 1
   - Keep note_text in the source/reference data.
   - Do not use raw note_text as model input yet.
   - Use structured note-derived features only.
   - Any note-derived variable must be checked for post-event leakage.
   - note_mentions_prosthetic_failure must NOT be blindly used as an input.

6. SYNTHETIC GENERATOR ORDER
   patient profile
       -> static valve characteristics
       -> latent deterioration phenotype
       -> baseline physiology
       -> longitudinal physiology/hemodynamics
       -> medication evolution
       -> note-derived signals
       -> event time
       -> censoring
       -> missingness

7. AFTER THE SYNTHETIC DATASET EXISTS
   Convert to model tensors:
       X_static       : [N, F_static]
       X_physiology   : [N, T, F_phys]
       X_medications  : [N, T, F_med]
       mask            : [N, T]
       duration        : [N]
       event           : [N]

8. MODEL VERSION 1
   X_static -> Valve MLP -> z_valve
   X_physiology -> GRU -> z_phys
   X_medications -> GRU -> z_med
   [z_phys, z_med] -> Temporal Fusion -> z_patient
   [z_patient, z_valve] -> Survival Head -> S(t)
"""
    path.write_text(text, encoding="utf-8")




## 2. Load the existing multimodal dataset

In [3]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Input file not found: {INPUT_PATH}')

df = pd.read_csv(INPUT_PATH)

required = {'Patient', 'Year'}
missing_required = required - set(df.columns)
if missing_required:
    raise ValueError(
        f'Missing required columns for patient-year structure: {sorted(missing_required)}'
    )

print('Shape:', df.shape)
print('Patients:', df['Patient'].nunique())
display(df.head())

Shape: (226, 82)
Patients: 117


,Patient,Year,note_count,note_types,note_services,signed_statuses,provider_types,provider_specialties,min_creation_year,max_creation_year,...,med__antiarrhythmic__present,med__insulin__present,medications_observed,Index_Date,Valve_Failure,notes_available,labs_available,medications_available,rich_lab_med,relative_to_index_year_UNVERIFIED
0,Patient_001,2022,1.0,Operative Report,Cardiac Surgery,Signed,Physician,Cardiac Surg,2022.0,2022.0,...,NaN,NaN,0,2022.0,1.0,1.0,0.0,0.0,0.0,0.0
1,Patient_001,2025,1.0,Progress Notes,NaN,Signed,Physician,Cardiology,2025.0,2025.0,...,NaN,NaN,0,2022.0,1.0,1.0,0.0,0.0,0.0,3.0
2,Patient_002,2017,1.0,Operative Report,Cardiac Surgery,Signed,Physician,NaN,2017.0,2017.0,...,NaN,NaN,0,2018.0,1.0,1.0,0.0,0.0,0.0,-1.0
3,Patient_002,2018,1.0,Progress Notes,NaN,Signed,Physician Assistant,NaN,2018.0,2018.0,...,NaN,NaN,0,2018.0,1.0,1.0,0.0,0.0,0.0,0.0
4,Patient_003,2014,1.0,Operative Report,Cardiac Surgery,Signed,Physician,NaN,2014.0,2014.0,...,NaN,NaN,0,2014.0,0.0,1.0,0.0,0.0,0.0,0.0


## 3. Build the column audit

In [4]:
audit = make_audit(df)
display(audit)

audit.to_csv(OUTDIR / '01_column_audit.csv', index=False)

,column,dtype,non_missing_n,missing_n,missing_pct,n_unique,modality,role,recommended_encoder,v1_action,leakage_or_design_note,mean,std,min,p25,median,p75,max
0,Patient,object,226,0,0.00,117,index,identifier,NONE,KEEP,"Used for grouping, ordering, censor/event timi...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Year,int64,226,0,0.00,25,index,time/index,NONE,KEEP,"Used for grouping, ordering, censor/event timi...",2017.469027,5.153762,2001.0,2014.0,2017.5,2021.0,2027.0
2,note_count,float64,160,66,29.20,3,notes,metadata,Optional / none,REVIEW,Mostly QC/availability metadata; not core clin...,1.343750,0.604458,1.0,1.0,1.0,2.0,3.0
3,note_types,object,160,66,29.20,6,notes,metadata,Optional / none,REVIEW,Mostly QC/availability metadata; not core clin...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,note_services,object,101,125,55.31,11,notes,metadata,Optional / none,REVIEW,Mostly QC/availability metadata; not core clin...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,notes_available,float64,223,3,1.33,1,notes,metadata,Optional / none,REVIEW,Mostly QC/availability metadata; not core clin...,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0
78,labs_available,float64,223,3,1.33,2,physiology,availability/QC,Mask / auxiliary,KEEP_AS_MASK,Useful for missingness/observation mask rather...,0.372197,0.484478,0.0,0.0,0.0,1.0,1.0
79,medications_available,float64,223,3,1.33,2,medications,metadata/QC,Mask / auxiliary,REVIEW,Can support observation masks or utilization c...,0.372197,0.484478,0.0,0.0,0.0,1.0,1.0
80,rich_lab_med,float64,223,3,1.33,2,qc,quality_flag,NONE,REVIEW,Dataset quality/availability flag.,0.372197,0.484478,0.0,0.0,0.0,1.0,1.0


## 4. Build the encoder map

In [5]:
encoder_map = make_encoder_map(audit)
display(encoder_map)

encoder_map.to_csv(OUTDIR / '02_encoder_map.csv', index=False)

,column,modality,role,recommended_encoder,v1_action,leakage_or_design_note
0,Patient,index,identifier,NONE,KEEP,"Used for grouping, ordering, censor/event timi..."
1,Year,index,time/index,NONE,KEEP,"Used for grouping, ordering, censor/event timi..."
2,note_count,notes,metadata,Optional / none,REVIEW,Mostly QC/availability metadata; not core clin...
3,note_types,notes,metadata,Optional / none,REVIEW,Mostly QC/availability metadata; not core clin...
4,note_services,notes,metadata,Optional / none,REVIEW,Mostly QC/availability metadata; not core clin...
...,...,...,...,...,...,...
77,notes_available,notes,metadata,Optional / none,REVIEW,Mostly QC/availability metadata; not core clin...
78,labs_available,physiology,availability/QC,Mask / auxiliary,KEEP_AS_MASK,Useful for missingness/observation mask rather...
79,medications_available,medications,metadata/QC,Mask / auxiliary,REVIEW,Can support observation masks or utilization c...
80,rich_lab_med,qc,quality_flag,NONE,REVIEW,Dataset quality/availability flag.


## 5. Proposed synthetic dataset schema

In [6]:
proposed_schema = proposed_schema_from_existing(audit)
display(proposed_schema)

proposed_schema.to_csv(OUTDIR / '03_proposed_synthetic_schema.csv', index=False)

,variable,modality,role,encoder,source,reason
0,Index_Date,index,time/index,NONE,FROM_MULTIMODAL,"Used for grouping, ordering, censor/event timi..."
1,Patient,index,identifier,NONE,FROM_MULTIMODAL,"Used for grouping, ordering, censor/event timi..."
2,Valve_Failure,outcome,target,NONE,FROM_MULTIMODAL_TARGET,Never use as encoder input.
3,Year,index,time/index,NONE,FROM_MULTIMODAL,"Used for grouping, ordering, censor/event timi..."
4,duration_months,outcome,survival_target,NONE,ADD_SYNTHETIC,Time from implant to event or censoring.
...,...,...,...,...,...,...
92,signed_statuses,notes,metadata,Optional / none,REVIEW_EXISTING,Mostly QC/availability metadata; not core clin...
93,time_since_implant_months,index,time,NONE,ADD_SYNTHETIC,Canonical longitudinal clock for all encoders.
94,valve_model,valve_static,static_categorical,Valve Encoder (MLP/embedding),ADD_SYNTHETIC,Valve-specific durability context.
95,valve_position,valve_static,static_categorical,Valve Encoder (MLP/embedding),ADD_SYNTHETIC,Anatomical context.


## 6. Dataset summary and leakage checks

In [7]:
summary = dataset_summary(df, audit)
print(json.dumps(summary, indent=2, ensure_ascii=False))

with open(OUTDIR / '04_dataset_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

leakage_view = audit[
    audit['v1_action'].isin(['EXCLUDE_OR_TEMPORALLY_FILTER', 'TARGET_ONLY'])
][['column', 'v1_action', 'leakage_or_design_note']]
display(leakage_view)

{
  "n_rows": 226,
  "n_columns": 82,
  "n_patients": 117,
  "year_min": 2001,
  "year_max": 2027,
  "modalities": {
    "physiology": 32,
    "medications": 23,
    "notes": 21,
    "index": 4,
    "outcome": 1,
    "qc": 1
  },
  "v1_actions": {
    "KEEP": 49,
    "REVIEW": 20,
    "KEEP_WITH_TEMPORAL_RULE": 6,
    "EXCLUDE_OR_TEMPORALLY_FILTER": 3,
    "KEEP_AS_MASK": 2,
    "KEEP_FOR_V2": 1,
    "TARGET_ONLY": 1
  },
  "potential_leakage_columns_present": [
    "note_mentions_redo",
    "note_mentions_prosthetic_failure",
    "Valve_Failure",
    "note_mentions_valve_in_valve"
  ],
  "Valve_Failure_counts": {
    "0.0": 183,
    "1.0": 40
  }
}


,column,v1_action,leakage_or_design_note
15,note_mentions_redo,EXCLUDE_OR_TEMPORALLY_FILTER,Potential label/post-event leakage. Use only i...
16,note_mentions_valve_in_valve,EXCLUDE_OR_TEMPORALLY_FILTER,Potential label/post-event leakage. Use only i...
17,note_mentions_prosthetic_failure,EXCLUDE_OR_TEMPORALLY_FILTER,Potential label/post-event leakage. Use only i...
76,Valve_Failure,TARGET_ONLY,Never use as encoder input.


## 7. Save next-step instructions

In [8]:
write_next_steps(OUTDIR / '05_next_steps.txt')

print('DONE')
print('Outputs saved to:', OUTDIR)
print('\nNext step: define the synthetic generator rules before training encoders.')

DONE
Outputs saved to: \mnt\data\synthetic_design_outputs

Next step: define the synthetic generator rules before training encoders.


## What comes next

The next notebook should define the actual synthetic patient generator:

1. patient-level static valve profile,
2. latent deterioration phenotype,
3. baseline physiology,
4. longitudinal lab/hemodynamic trajectories,
5. medication trajectories,
6. note-derived structured signals,
7. event time and censoring,
8. realistic missingness.

Only after that should we build the encoders.

In [9]:
print(OUTDIR.resolve())

C:\mnt\data\synthetic_design_outputs
